# MorphIQ SDK for Python — Regime analysis

Examine a ticker's price history and the API's latest regime assessment.

**Run the workspace cell once to display the controls.** Choose a ticker and daily,
1-minute, 5-minute, 15-minute, 30-minute or hourly prices, then click **Analyze regime**.
That one button downloads Yahoo history, validates the prices, submits one MorphIQ
request and displays the complete analysis. No additional cells or load buttons are needed.
Changing the ticker or interval clears the previous selection; click **Analyze regime**
to process the new selection. Once complete, **Show analysis** redraws the saved response
without another request. Setup and control changes make no network requests.
Each analysis request may consume API credits.

The workspace keeps the latest 512 prices in memory, using the same split-adjusted
Yahoo closes as option pricing, without dividend adjustment. Daily mode uses completed
daily closes; intraday mode uses regular-session bars and shows their timestamps and
timezone. The latest intraday bar may still be forming or delayed. If Yahoo cannot supply 512 valid prices, the notebook explains the problem without
sending a MorphIQ request.

Docker includes everything required. Outside Docker, install this repository with
`pip install -e '.[notebooks]'`. Set `MORPHIQ_API_KEY` before starting, or enter it
at the masked prompt. [Get API access](https://developers.morphiqlabs.com).

[Back to start](../start-here.ipynb)


In [ ]:
import math
import os
from datetime import date, timedelta
from getpass import getpass

import ipywidgets as widgets
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
from IPython.display import display

from morphiq_sdk import APIError, MorphIQ, RegimeRequest, ResponseError, TransportError

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

ticker = widgets.Text(
    value="SPY",
    description="Ticker",
    continuous_update=False,
    layout=widgets.Layout(width="260px"),
)
price_interval = widgets.Dropdown(
    options=[
        ("Daily close", "1d"),
        ("1 minute", "1m"),
        ("5 minutes", "5m"),
        ("15 minutes", "15m"),
        ("30 minutes", "30m"),
        ("1 hour", "60m"),
    ],
    value="1d",
    description="Price interval",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "initial"},
)
analyze_button = widgets.Button(
    description="Analyze regime · 1 request",
    button_style="primary",
    layout=widgets.Layout(width="260px"),
)
history_output = widgets.Output()
analysis_output = widgets.Output()
analysis_status = widgets.Label(value="Choose a ticker and interval, then Analyze regime.")
loaded_closes = None
loaded_ticker = None
loaded_interval = None
last_response = None


def invalidate_history(change):
    global loaded_closes, loaded_ticker, loaded_interval, last_response
    loaded_closes = loaded_ticker = loaded_interval = last_response = None
    analyze_button.disabled = False
    history_output.outputs = ()
    analysis_output.outputs = ()
    analysis_status.value = "Choose a ticker and interval, then Analyze regime."
    analyze_button.description = "Analyze regime · 1 request"


ticker.observe(invalidate_history, names="value")
price_interval.observe(invalidate_history, names="value")


def yahoo_history_options(interval):
    if interval == "1d":
        end = date.today()
        return {"interval": interval, "start": end - timedelta(days=3 * 366), "end": end}
    # Fetch enough regular-session bars for 512 prices within Yahoo's lookback limits.
    periods = {"1m": "5d", "5m": "1mo", "15m": "60d", "30m": "60d", "60m": "6mo"}
    return {"interval": interval, "period": periods[interval]}


def price_timestamp(timestamp, interval):
    if interval == "1d":
        return str(timestamp.date())
    return f"{timestamp.isoformat()} ({timestamp.tz or 'timezone not reported'})"


def price_history_figure(closes, symbol, interval):
    fig, ax = plt.subplots(figsize=(9, 3), layout="constrained")
    if interval == "1d":
        ax.plot(closes.index, closes.values, color="#2459a6")
    else:
        # Break only the drawing at gaps; preserve every input price and timestamp.
        spacing = pd.Timedelta(minutes=int(interval.removesuffix("m")))
        segments = closes.index.to_series().diff().gt(spacing).cumsum()
        for _, segment in closes.groupby(segments):
            ax.plot(segment.index, segment.values, color="#2459a6", marker=".", markersize=2)
    locator = mdates.AutoDateLocator(minticks=3, maxticks=7, tz=closes.index.tz)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator, tz=closes.index.tz))
    ax.set(
        title=f"{symbol} · {interval} input price history",
        ylabel="Price",
        xlabel=f"Time ({closes.index.tz or 'timezone not reported'})"
        if interval != "1d"
        else "Date",
    )
    ax.grid(alpha=0.25)
    return fig


def use_history(history, symbol, interval="1d"):
    global loaded_closes, loaded_ticker, loaded_interval, last_response
    loaded_closes = loaded_ticker = loaded_interval = last_response = None
    if history is None or history.empty:
        raise ValueError("Yahoo returned no history. Check the ticker.")
    closes = history["Close"].tail(512).copy()
    if len(closes) != 512 or not all(math.isfinite(value) and value > 0 for value in closes):
        raise ValueError(
            f"Need 512 valid positive closes; Yahoo returned {len(closes)} bars in the window. "
            "Try another ticker or interval. "
            "Missing or invalid closes in returned bars are not filled or dropped."
        )
    if not closes.index.is_monotonic_increasing or not closes.index.is_unique:
        raise ValueError("Price dates must be unique and ordered oldest to newest.")
    analysis_status.value = "Ready to analyze."
    analyze_button.description = "Analyze regime · 1 request"
    loaded_closes, loaded_ticker, loaded_interval = closes, symbol, interval
    with history_output.hold_sync():
        history_output.outputs = ()
        history_output.append_stdout(
            f"{symbol}: {len(closes)} Yahoo {interval} closes\n"
            f"{price_timestamp(closes.index[0], interval)} – "
            f"{price_timestamp(closes.index[-1], interval)}\n"
            f"Latest close: {closes.iloc[-1]:.6g}. Split-adjusted, without dividend adjustment.\n"
        )
        if interval != "1d":
            history_output.append_stdout(
                "The latest returned bar may still be forming, and Yahoo data may be delayed.\n"
            )
        fig = price_history_figure(closes, symbol, interval)
        try:
            history_output.append_display_data(fig)
        finally:
            plt.close(fig)


def load_history():
    symbol = ticker.value.strip().upper()
    if not symbol:
        raise ValueError("Enter a ticker.")
    selected_interval = price_interval.value
    history = yf.download(
        symbol,
        **yahoo_history_options(selected_interval),
        prepost=False,
        ignore_tz=False,
        auto_adjust=False,
        keepna=False,  # Use yfinance's standard empty-bar cleanup.
        multi_level_index=False,
        progress=False,
        threads=False,
    )
    use_history(history, symbol, selected_interval)


def render_regime(response):
    # Sync the complete result together, independently of callback output capture.
    with analysis_output.hold_sync():
        analysis_output.outputs = ()
        analysis_output.append_stdout(
            f"{loaded_ticker} · {loaded_interval} regime at the final supplied close "
            f"({price_timestamp(loaded_closes.index[-1], loaded_interval)})\n"
        )
        rows = []
        heads = [
            ("Decision", response.result.decision),
            ("Direction", response.result.direction),
            ("State", response.result.state),
        ]
        for name, head in heads:
            rows.append(
                {
                    "Head": name,
                    "Status": head.status,
                    "Target": head.basis.target,
                    "Result": head.prediction.label
                    if head.status == "available"
                    else head.reason_code,
                }
            )
        analysis_output.append_display_data(pd.DataFrame(rows).set_index("Head"))
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
        for ax, (name, head) in zip(axes, heads, strict=True):
            ax.set_title(name)
            if head.status == "available":
                labels = [score.label for score in head.prediction.scores]
                scores = [score.value for score in head.prediction.scores]
                ax.barh(labels, scores, color="#2459a6")
                ax.set_xlim(0, 1)
                ax.set_xlabel("Uncalibrated score")
            else:
                ax.text(
                    0.5,
                    0.5,
                    f"{head.status}\n{head.reason_code}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    wrap=True,
                )
                ax.set_axis_off()
        try:
            fig.tight_layout()
            analysis_output.append_display_data(fig)
        finally:
            plt.close(fig)
        metadata = response.metadata
        consumed = metadata.consumed_price_range
        charge = metadata.fcu_charged
        analysis_output.append_stdout(
            f"Consumed {metadata.price_count} of {metadata.received_price_count} supplied prices: "
            f"indices [{consumed.start_index}, {consumed.end_index_exclusive}).\n"
            f"Model: {metadata.provenance.model_id} ({metadata.provenance.model_status})\n"
            f"Request ID: {metadata.request_id}\n"
            f"CU charged: {charge if charge is not None else 'not reported'}\n"
        )
        details = {
            "Model SHA-256": metadata.provenance.model_sha256,
            "Preprocessing": metadata.provenance.preprocessing,
            "Reference teacher executed": metadata.provenance.teacher_executed,
            "Return type / unit": f"{metadata.return_type} / {metadata.return_unit}",
            "Selected input index": metadata.input_price_index,
            "Engine(s)": "; ".join(
                f"{e.name} {e.version} ({e.revision})" for e in metadata.engines
            ),
        }
        analysis_output.append_display_data(
            pd.DataFrame(details.items(), columns=["Provenance", "Value"])
        )


def analyze_regime(_):
    global last_response
    ticker.disabled = price_interval.disabled = analyze_button.disabled = True
    try:
        if (
            loaded_closes is None
            or loaded_ticker != ticker.value.strip().upper()
            or loaded_interval != price_interval.value
        ):
            analysis_status.value = "Loading Yahoo history…"
            try:
                load_history()
            except Exception as error:
                invalidate_history(None)
                analysis_status.value = f"History lookup failed: {error}"
                return
        analysis_status.value = (
            "Analyzing…" if last_response is None else "Displaying saved analysis…"
        )
        if last_response is None:
            # Bar labels do not establish the exact final observation time; omit as_of.
            request = RegimeRequest(prices=loaded_closes.tolist())
            with MorphIQ() as client:
                last_response = client.regime(request)
        render_regime(last_response)
        analysis_status.value = "Analysis complete. Showing it again uses the saved response."
    except APIError as error:
        charge = error.metadata.fcu_charged if error.metadata else None
        analysis_output.outputs = ()
        analysis_output.append_stdout(
            f"HTTP {error.status_code}: {error.code}: {error.message}\n"
            f"Request ID: {error.request_id}; "
            f"CU charged: {charge if charge is not None else 'not reported'}\n"
        )
        analysis_status.value = "Request failed. No automatic retry was made."
    except (TransportError, ResponseError) as error:
        analysis_output.outputs = ()
        analysis_output.append_stdout(
            "No usable response. Execution and charges may be unknown; no retry was made.\n"
        )
        if isinstance(error, ResponseError):
            analysis_output.append_stdout(f"Request ID: {error.request_id}\n")
        analysis_status.value = "Request failed. No automatic retry was made."
    except Exception as error:
        analysis_status.value = f"Display failed ({type(error).__name__})."
        if last_response is not None:
            analysis_status.value += " Click Show analysis to redraw the saved response."
            analysis_output.append_stdout(
                f"Saved response: {last_response.metadata.request_id}. No new request is needed.\n"
            )
    finally:
        ticker.disabled = price_interval.disabled = analyze_button.disabled = False
        analyze_button.description = (
            "Show analysis" if last_response is not None else "Analyze regime · 1 request"
        )


analyze_button.on_click(analyze_regime)
display(
    widgets.VBox(
        [
            widgets.HBox(
                [ticker, price_interval, analyze_button],
                layout=widgets.Layout(flex_flow="row wrap"),
            ),
            history_output,
            analysis_status,
            analysis_output,
        ]
    )
)

### Reading the assessment

The assessment is for the final supplied observation. The price chart is input history;
it does not show historical regime classifications. Intraday lines break across
overnight, weekend and missing-bar gaps; the request still includes the original
ordered prices, including changes between sessions. V1 takes the latest 512 prices
and consumes their latest 507; the response reports the exact consumed range.
Changing the interval changes the time span represented by those prices. The API
receives the ordered prices, without a sampling-interval parameter. Daily dates and
intraday bar labels do not establish exact final observation times, so `as_of` is omitted.

Each head has its own status. **Available** includes a prediction; **not applicable**
and **unavailable** include a reason instead. `unresolved` is a valid available decision,
and `NEUTRAL` is a valid direction. `LOWER_VOL` and `HIGHER_VOL` compare the reference
model's two states, not historical volatility norms or price direction.

The bars show **uncalibrated softmax scores**, not calibrated probabilities, p-values
or confidence intervals. Do not multiply scores across heads. The basis describes
what is predicted; it does not imply that reference fitting ran for your request.
Keep the model provenance and request ID with any interpretation.

Missing CU means unreported, not zero. No automatic retries occur. Clear outputs before
sharing and shut down the kernel when finished. Expand the hidden code cell to inspect the implementation.
